# Replication code (revised): meta-analysis of connectedness estimates
Colab version of `meta_analysis.py`. Reproduces pooled effects, heterogeneity, moderator analyses, publication-bias tests and robustness checks (Tables 3–8, Table 6 in full).

**Before running:** put `meta_estimates_203.csv` and `review_sample_257.csv` in `MyDrive/green_finance_meta/data/` (or upload them via the Colab file panel after the setup cell). Run this notebook **first**; the supplementary notebook reads `output/meta_results.json`.

Changes from the first version: (i) REML τ² with Hartung–Knapp intervals (DL and Paule–Mandel as sensitivity); (ii) within-study aggregation = mean of a study's estimates in a state; (iii) adjusted TCI capped at 100; (iv) "excluding Dai et al. (2023)" no longer drops Dai et al. (2024); (v) Mensi et al. (2025) added; (vi) Table 3, Table 4 median/tail rows and Table 6 columns (1)–(3) produced.

## 1. Setup (mount Drive, folders)

In [13]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Project folder in Google Drive (change if you like). Both notebooks share it,
# so output/meta_results.json written by notebook 1 is visible to notebook 2.
ROOT = '/content/drive/MyDrive/green_finance_meta'
for sub in ['data', 'output', 'figures', 'tables']:
    os.makedirs(f'{ROOT}/{sub}', exist_ok=True)
os.chdir(ROOT)

needed = ['data/meta_estimates_203.csv', 'data/review_sample_257.csv']
missing = [f for f in needed if not os.path.exists(f)]
print('Working directory:', os.getcwd())
print('Missing files:', missing if missing else 'none')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/green_finance_meta
Missing files: ['data/meta_estimates_203.csv', 'data/review_sample_257.csv']


## 2. Data preparation and helper functions

In [14]:
import pandas as pd,numpy as np,json,os,statsmodels.api as sm,statsmodels.formula.api as smf
from scipy import stats,optimize
os.makedirs('output',exist_ok=True)
e=pd.read_csv('/content/meta_estimates_203.csv')
d=e.rename(columns={'study_id':'study','TCI_adjusted_pct':'TCI','TCI_reported_pct':'TCI_pct','tail_quantile':'tail'}).copy()
d['G']=d.Green_asset_class; d['R']=d.Region; d['M']=d.Method_family.replace({'BK-frequency':'DY-rolling','Other':'DY-rolling','QVAR-frequency':'QVAR','TVP-VAR-BK':'TVP-VAR'})
d['Freq_band']=d.Freq_band.fillna('Full'); d['vol']=(d.Connectedness_type=='Volatility').astype(int); d['hm']=(d.Connectedness_type=='Higher-moment').astype(int)
d['log_n']=np.log(d.n_vars); d['crisis']=(d.Crisis_subsample.fillna('None')!='None').astype(int)
d['weak']=d.Value_source.isin(['Figure-read','Computed','Text']).astype(int)
d['TCI']=d.TCI.clip(upper=100)            # adjusted TCI cannot exceed 100 (Qiao et al. 2024 reported value exceeds (N-1)/N)
def _ym(s):
    import re
    if pd.isna(s): return np.nan
    m=re.match(r'(\d{4})(?:-(\d{2}))?',str(s)); return int(m.group(1))+((int(m.group(2))-1)/12 if m.group(2) else 0.5)
d['y1']=d.Sample_end.map(_ym)
B=d[d.baseline].copy()

inv=lambda x:100/(1+np.exp(-x))
def prep(X,s=1):
    p=(X.TCI/100).clip(0.005,0.995); y=np.log(p/(1-p)); v=1/((X.T_use/s)*p*(1-p)); return np.asarray(y,float),np.asarray(v,float)
def dl(y,v):
    w=1/v;k=len(y);yb=np.sum(w*y)/np.sum(w);Q=np.sum(w*(y-yb)**2);df=k-1
    C=np.sum(w)-np.sum(w**2)/np.sum(w);t2=max(0,(Q-df)/C) if k>1 else 0
    wr=1/(v+t2);mu=np.sum(wr*y)/np.sum(wr);se=np.sqrt(1/np.sum(wr));I2=max(0,(Q-df)/Q)*100 if Q>0 else 0
    tcrit=stats.t.ppf(0.975,max(k-2,1)); pi=(mu-tcrit*np.sqrt(t2+se**2),mu+tcrit*np.sqrt(t2+se**2))
    return dict(k=k,Q=Q,df=df,p=stats.chi2.sf(Q,df) if df>0 else np.nan,I2=I2,t2=t2,mu=mu,se=se,lo=mu-1.96*se,hi=mu+1.96*se,pilo=pi[0],pihi=pi[1])
def tau2_reml(y,v):
    def nll(t2):
        w=1/(v+t2); mu=np.sum(w*y)/np.sum(w); return 0.5*(np.sum(np.log(v+t2))+np.log(np.sum(w))+np.sum(w*(y-mu)**2))
    return float(optimize.minimize_scalar(nll,bounds=(0,20),method='bounded',options={'xatol':1e-8}).x)
def tau2_pm(y,v):
    f=lambda t2:np.sum((1/(v+t2))*(y-np.sum(y/(v+t2))/np.sum(1/(v+t2)))**2)-(len(y)-1)
    return optimize.brentq(f,0,50) if f(0)>0 else 0.0
def re(y,v,method='REML'):
    """Random-effects pooling. Wald CI, Hartung-Knapp (modified, q>=1) CI, and prediction interval (t_{k-2})."""
    k=len(y); base=dl(y,v)
    t2={'REML':tau2_reml,'PM':tau2_pm,'DL':lambda a,b:dl(a,b)['t2']}[method](y,v) if k>1 else 0.0
    w=1/(v+t2); mu=np.sum(w*y)/np.sum(w); se=np.sqrt(1/np.sum(w))
    q=np.sum(w*(y-mu)**2)/(k-1) if k>1 else 1.0; se_hk=np.sqrt(max(q,1.0)/np.sum(w))
    tk=stats.t.ppf(0.975,max(k-1,1)); tp=stats.t.ppf(0.975,max(k-2,1))
    return dict(k=k,Q=base['Q'],df=base['df'],p=base['p'],I2=base['I2'],t2=t2,mu=mu,se=se,lo=mu-1.96*se,hi=mu+1.96*se,
                hklo=mu-tk*se_hk,hkhi=mu+tk*se_hk,pilo=mu-tp*np.sqrt(t2+se**2),pihi=mu+tp*np.sqrt(t2+se**2))
def mreg(y,v,X):
    # mixed-effects meta-regression, MM tau2, modified Knapp-Hartung
    X=np.asarray(X,float); k,p=X.shape; W=np.diag(1/v)
    XtWX=X.T@W@X; b=np.linalg.solve(XtWX,X.T@W@y); e_=y-X@b; QE=float(e_@W@e_)
    P=W-W@X@np.linalg.solve(XtWX,X.T@W); t2=max(0,(QE-(k-p))/np.trace(P))
    Ws=np.diag(1/(v+t2)); A=X.T@Ws@X; b=np.linalg.solve(A,X.T@Ws@y); e_=y-X@b
    s2=float(e_@Ws@e_)/(k-p); cov=np.linalg.inv(A)*max(s2,1); se=np.sqrt(np.diag(cov))
    return b,se,t2,QE,k-p,cov
def pooled_rows(df,agg='mean'):
    g=df.groupby('study').agg(TCI=('TCI','mean'),T_use=('T_use','first')).reset_index(); return prep(g)

## 3. Analyses

In [15]:
R = {}
yb,vb=prep(B); base=re(yb,vb,'REML'); R['overall']=base
R['overall_dl']=re(yb,vb,'DL'); R['overall_pm']=re(yb,vb,'PM')
R['loo']=[inv(re(np.delete(yb,i),np.delete(vb,i))['mu']) for i in range(len(yb))]
R['T10']=re(*prep(B,10)); R['T100']=re(*prep(B,100))
R['Astar']=re(*prep(B[B.study.isin(d[d.Value_source.notna()].study)&B['Study (Author Year)'].notna()&B.study.isin(pd.read_csv('/content/review_sample_257.csv').merge(e[['EID']].drop_duplicates(),on='EID').query("ABDC=='A*'").EID.map(dict(zip(e.EID,e.study_id))))]))
R['Aonly']=re(*prep(B[~B.study.isin(pd.read_csv('/content/review_sample_257.csv').merge(e[['EID']].drop_duplicates(),on='EID').query("ABDC=='A*'").EID.map(dict(zip(e.EID,e.study_id))))]))
R['noDai']=re(*prep(B[B['Study (Author Year)']!='Dai et al. (2023)']))

In [16]:
# ---- Table 3 counts
R['counts']=dict(n_est=len(d),n_studies=int(d.study.nunique()),full=int(((d.Freq_band=='Full')&(d['tail']==0)).sum()),tail=int(d['tail'].sum()),band=int((d.Freq_band!='Full').sum()),tail_and_band=int(((d.Freq_band!='Full')&(d['tail']==1)).sum()),
                 full_studies=int(d[(d.Freq_band=='Full')&(d['tail']==0)].study.nunique()),tail_studies=int(d[d['tail']==1].study.nunique()),band_studies=int(d[d.Freq_band!='Full'].study.nunique()))

In [17]:
# ---- median / tail pooled (study-mean rule)
R['median']=re(*pooled_rows(d[d.Quantile==0.5])); R['tailpool']=re(*pooled_rows(d[d['tail']==1]))
R['lower']=re(*pooled_rows(d[(d['tail']==1)&(d.Quantile<=0.10)])); R['upper']=re(*pooled_rows(d[(d['tail']==1)&(d.Quantile>=0.90)]))

In [18]:
# ---- subgroup moderators
B2=B.copy()
B2['Estimator']=B2.M.map({'DY-rolling':'Diebold–Yilmaz','TVP-VAR':'TVP-VAR','QVAR':'Quantile VAR (median)'})
B2['Region']=B2.R.map({'Global':'Global / multi-country','China':'China','Europe':'Europe','US':'United States'})
B2['Spillover type']=B2.Connectedness_type.map({'Return':'Return','Volatility':'Volatility'}).fillna('Other')
B2['Sample includes 2020+']=np.where(B2.y1>=2020,'Yes','No')
B2['System size']=np.where(B2.n_vars<=6,'Small (N ≤ 6)','Large (N > 6)')
B2['Publication year']=np.where(B2.pubyear<=2022,'2017–2022','2023–2026')
B2['Asset class']=B2.G.map({'GreenBond':'Green bonds','CleanEnergy':'Clean energy','Carbon':'Carbon','ESG/Sustainability':'ESG / sustainability','GreenCrypto':'Green crypto'})
sub={}
for mod in ['Asset class','Estimator','Region','Spillover type','Sample includes 2020+','System size','Publication year']:
    levels=list(B2[mod].value_counts().index)
    X=np.column_stack([np.ones(len(B2))]+[(B2[mod]==l).astype(float) for l in levels[1:]])
    b,se,t2,QE,df,cov=mreg(yb,vb,X)
    bb=b[1:];cc=cov[1:,1:]; QM=float(bb@np.linalg.solve(cc,bb)); pm=stats.f.sf(QM/len(bb),len(bb),df)
    mm0=mreg(yb,vb,np.ones((len(yb),1)))[2]; R2=max(0,(mm0-t2)/mm0)*100 if mm0>0 else 0
    grp=[]
    for l in levels:
        m=(B2[mod]==l).values; g=re(yb[m],vb[m]); grp.append((l,g['k'],inv(g['mu']),inv(g['hklo']),inv(g['hkhi']),g['I2'],g['t2'],inv(g['pilo']),inv(g['pihi']),g['Q'],g['df']))
    sub[mod]=dict(QM=QM,df=len(bb),p=pm,R2=R2,groups=grp)
R['sub']=sub

In [19]:
# ---- study-level multivariable meta-regression
Xdf=pd.DataFrame({'const':1,'logN':np.log(B2.n_vars),'TVP':(B2.M=='TVP-VAR')*1.,'QV':(B2.M=='QVAR')*1.,'vol':(B2.Connectedness_type=='Volatility')*1.,
 'GB':(B2.G=='GreenBond')*1.,'CA':(B2.G=='Carbon')*1.,'ESG':(B2.G=='ESG/Sustainability')*1.,'GC':(B2.G=='GreenCrypto')*1.,
 'CN':(B2.R=='China')*1.,'EU':(B2.R=='Europe')*1.,'US':(B2.R=='US')*1.,'post20':(B2.y1>=2020)*1.})
b,se,t2,QE,df,cov=mreg(yb,vb,Xdf.values); mm0=mreg(yb,vb,np.ones((len(yb),1)))[2]
R['mreg']=dict(names=list(Xdf.columns),b=b.tolist(),se=se.tolist(),p=(2*stats.t.sf(np.abs(b/se),df)).tolist(),t2=t2,R2=max(0,(mm0-t2)/mm0)*100,QE=QE,df=df,I2res=max(0,(QE-df)/QE)*100,k=len(yb))

In [20]:
# ---- publication bias (DL-based trim-and-fill; Egger/PET share one regression)
def egger(y,v):
    se=np.sqrt(v); r=sm.OLS(y/se,sm.add_constant(1/se)).fit(); return r.params[0],r.bse[0],r.pvalues[0]
def pet(y,v,sq=False):
    se=np.sqrt(v); x=v if sq else se; r=sm.WLS(y,sm.add_constant(x),weights=1/v).fit(); return r.params[0],r.bse[0],r.pvalues[0],r.params[1],r.pvalues[1]
def begg(y,v):
    w=1/v;mu=np.sum(w*y)/np.sum(w);vs=v-1/np.sum(w);z=(y-mu)/np.sqrt(vs);return stats.kendalltau(z,v)
def trimfill(y,v,side=None,centre='FE'):
    """Duval-Tweedie L0 trim-and-fill. Standard (original) algorithm centres on the fixed-effect mean;
    centre='RE' re-centres on the random-effects mean, which is unstable when tau^2 is very large."""
    k=len(y)
    if side is None: side='left' if egger(y,v)[0]>0 else 'right'
    sgn=1 if side=='left' else -1; k0=0
    for _ in range(100):
        idx=np.argsort(y*sgn); yy=y[idx][:k-k0] if k0>0 else y; vv=v[idx][:k-k0] if k0>0 else v
        mu=np.sum(yy/vv)/np.sum(1/vv) if centre=='FE' else dl(yy,vv)['mu']
        dev=(y-mu)*sgn; r=stats.rankdata(np.abs(dev)); Tn=np.sum(r[dev>0])
        L0=max(0,int(round((4*Tn-k*(k+1))/(2*k-1))))
        if L0==k0: break
        k0=L0
    if k0==0: return 0,re(y,v)
    idx=np.argsort(y*sgn)[::-1][:k0]; return k0,re(np.r_[y,2*mu-y[idx]],np.r_[v,v[idx]])
pb={}
for lab,s in [('T',1),('T/10',10)]:
    y,v=prep(B,s); e1=egger(y,v); bt=begg(y,v); tf=trimfill(y,v); tfre=trimfill(y,v,centre='RE'); p1=pet(y,v); p2=pet(y,v,True)
    pb[lab]=dict(egger=e1,begg=(bt[0],bt[1]),tf_k0=tf[0],tf_k0_REcentre=tfre[0],tf_mu_REcentre=inv(tfre[1]['mu']),tf_mu=inv(tf[1]['mu']),tf_lo=inv(tf[1]['lo']),tf_hi=inv(tf[1]['hi']),pet=p1,peese=p2)
R['pb']=pb

In [21]:
# ---- within-study tail premium
m=d[d.Quantile==0.5]; t=d[d['tail']==1]; ws=[]
for s in sorted(set(m.study)&set(t.study)):
    ws.append((s,t[t.study==s].TCI.mean()-m[m.study==s].TCI.mean(),t[t.study==s].TCI_pct.mean()-m[m.study==s].TCI_pct.mean()))
wa=np.array([w[1] for w in ws]); wr=np.array([w[2] for w in ws]); tc_=stats.t.ppf(.975,len(wa)-1)
R['tailprem']=dict(k=len(ws),adj_mean=wa.mean(),adj_lo=wa.mean()-tc_*wa.std(ddof=1)/np.sqrt(len(wa)),adj_hi=wa.mean()+tc_*wa.std(ddof=1)/np.sqrt(len(wa)),
 raw_mean=wr.mean(),raw_min=wr.min(),raw_max=wr.max(),pos=int((wr>0).sum()),wilcoxon_p=stats.wilcoxon(wr).pvalue)

In [22]:
# ---- net-role proportions (Wilson)
def wil(x,n):
    if n==0: return (np.nan,np.nan,np.nan)
    z=1.96;p=x/n;den=1+z*z/n;c=(p+z*z/(2*n))/den;h=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den;return p*100,(c-h)*100,(c+h)*100
roles={}
for g,x in B2.groupby('Asset class'):
    tr=int((x.Green_net_role=='Transmitter').sum()); rc=int((x.Green_net_role=='Receiver').sum()); roles[g]=dict(tr=tr,rc=rc,recv_share=wil(rc,tr+rc))
R['roles']=roles

In [23]:
# ---- estimate-level regressions (Table 6 cols 1-3) and robustness (Table 8)
d2=d.copy()
f='TCI ~ tail + C(Freq_band, Treatment("Full")) + vol + hm + C(M, Treatment("DY-rolling")) + log_n + crisis + C(G, Treatment("CleanEnergy")) + C(R, Treatment("Global"))'
def fit(df,formula=f,dep='TCI',w=None):
    ff=formula.replace('TCI ~',dep+' ~')
    return (smf.wls(ff,df,weights=w) if w is not None else smf.ols(ff,df)).fit(cov_type='cluster',cov_kwds={'groups':df.study})
def tailc(df,formula=f,dep='TCI',w=None):
    r=fit(df,formula,dep,w); return r.params['tail'],r.bse['tail'],r.pvalues['tail'],r.params.get('log_n',np.nan)
f_noband=f.replace(' + C(Freq_band, Treatment("Full"))','')
c1=fit(d2); c2=fit(d2[d2.Freq_band=='Full'],f_noband); dd=d2[(d2.weak==0)&(d2['Study (Author Year)']!='Dai et al. (2023)')]; c3=fit(dd)
def pack(r,df): return dict(params=r.params.to_dict(),bse=r.bse.to_dict(),p=r.pvalues.to_dict(),n=int(r.nobs),studies=int(df.study.nunique()),r2=float(r.rsquared))
R['t6']=dict(c1=pack(c1,d2),c2=pack(c2,d2[d2.Freq_band=='Full']),c3=pack(c3,dd))
rob=[('Baseline: all estimates, OLS, clustered SE',)+tailc(d2),
     ('Full-sample estimates only (no frequency bands)',)+tailc(d2[d2.Freq_band=='Full'],f_noband),
     ('Table values only, excluding Dai et al. (2023)',)+tailc(dd),
     ('Unadjusted TCI as dependent variable',)+tailc(d2,dep='TCI_pct'),
     ('Equal weight per study (weights 1/estimates)',)+tailc(d2,w=1/d2.groupby('study').TCI.transform('size')),
     ('Study fixed effects (within-study variation)',)+tailc(d2,'TCI ~ tail + C(Freq_band, Treatment("Full")) + vol + hm + C(M, Treatment("DY-rolling")) + crisis + C(study)'),
     ('Excluding two-variable systems and monthly data',)+tailc(d2[(d2.n_vars>2)&(d2.Data_frequency!='Monthly')])]
lo=[tailc(d2[d2.study!=s])[0] for s in d2.study.unique()]
R['rob']=rob; R['rob_loo']=(min(lo),max(lo))
# lower vs upper tail
d2['lowT']=((d2.Quantile<=0.10)&(d2['tail']==1)).astype(int); d2['upT']=((d2.Quantile>=0.90)&(d2['tail']==1)).astype(int)
r5=fit(d2,f.replace('tail +','lowT + upT +')); wt=r5.wald_test('lowT = upT',scalar=True)
lo_=d[(d.Quantile<=0.10)&(d['tail']==1)]; up_=d[(d.Quantile>=0.90)&(d['tail']==1)]
both=sorted(set(lo_.study)&set(up_.study)); diff=np.array([up_[up_.study==s].TCI.mean()-lo_[lo_.study==s].TCI.mean() for s in both])
R['lowup']=dict(low=float(r5.params['lowT']),low_se=float(r5.bse['lowT']),up=float(r5.params['upT']),up_se=float(r5.bse['upT']),wald_p=float(wt.pvalue),k_both=len(both),diff=float(diff.mean()),n_up_gt=int((diff>0).sum()),wil_p=float(stats.wilcoxon(diff).pvalue))
# wild cluster bootstrap (Rademacher) for tail
rng=np.random.default_rng(1); r0=smf.ols(f,d2).fit(); Xm=r0.model.exog; ti=list(r0.params.index).index('tail')
rr=smf.ols(f.replace('tail + ',''),d2).fit(); fit0=rr.fittedvalues.values; res0=rr.resid.values
cl=d2.study.values; ucl=np.unique(cl); bs=[]
for _ in range(1999):
    g=dict(zip(ucl,rng.choice([-1,1],len(ucl)))); ys=fit0+res0*np.array([g[c] for c in cl]); bs.append(np.linalg.lstsq(Xm,ys,rcond=None)[0][ti])
R['wcb_p']=float(np.mean(np.abs(np.array(bs))>=abs(r0.params['tail'])))
json.dump(R,open('output/meta_results.json','w'),default=lambda o: float(o) if isinstance(o,(np.floating,np.integer)) else str(o))
B2.to_csv('output/baseline_estimates.csv',index=False)
fmt=lambda r:f"{inv(r['mu']):.1f} (HK {inv(r['hklo']):.1f}-{inv(r['hkhi']):.1f}; Wald {inv(r['lo']):.1f}-{inv(r['hi']):.1f}) PI {inv(r['pilo']):.1f}-{inv(r['pihi']):.1f} tau2 {r['t2']:.3f} I2 {r['I2']:.1f} Q {r['Q']:.0f}"
for k_ in ['overall','overall_dl','overall_pm','T10','T100','Astar','Aonly','noDai','median','tailpool','lower','upper']: print(k_,R[k_]['k'],fmt(R[k_]))
print('loo',min(R['loo']),max(R['loo'])); print('counts',R['counts']); print('tailprem',R['tailprem']); print('lowup',R['lowup'])
for k_,v_ in sub.items(): print(k_,round(v_['QM'],2),v_['df'],round(v_['p'],3),'R2',round(v_['R2'],1),[(g[0],g[1],round(g[2],1),f'{g[3]:.1f}-{g[4]:.1f}') for g in v_['groups']])
print('mreg',{n:(round(b_,2),round(s_,2),round(p_,3)) for n,b_,s_,p_ in zip(R['mreg']['names'],R['mreg']['b'],R['mreg']['se'],R['mreg']['p'])},'R2',R['mreg']['R2'],R['mreg']['I2res'])
print('pb',pb); print('roles',roles)
for c_ in ['c1','c2','c3']: print(c_,R['t6'][c_]['n'],R['t6'][c_]['studies'],round(R['t6'][c_]['r2'],3))
for r_ in rob: print([r_[0]]+[round(x,2) for x in r_[1:]])
print('rob loo',R['rob_loo'],'wcb p',R['wcb_p'])

overall 64 56.3 (HK 49.2-63.2; Wald 49.4-63.0) PI 11.6-92.7 tau2 1.290 I2 99.6 Q 15742
overall_dl 64 56.4 (HK 49.3-63.2; Wald 51.6-61.1) PI 20.8-86.4 tau2 0.625 I2 99.6 Q 15742
overall_pm 64 56.3 (HK 49.2-63.2; Wald 49.4-63.1) PI 11.5-92.8 tau2 1.301 I2 99.6 Q 15742
T10 64 56.7 (HK 49.8-63.3; Wald 50.2-62.9) PI 13.9-91.4 tau2 1.078 I2 96.0 Q 1574
T100 64 57.2 (HK 51.7-62.6; Wald 52.1-62.1) PI 27.4-82.5 tau2 0.388 I2 60.0 Q 157
Astar 43 55.3 (HK 45.2-64.9; Wald 45.6-64.7) PI 7.9-94.7 tau2 1.706 I2 99.7 Q 12468
Aonly 21 58.4 (HK 50.5-65.9; Wald 51.0-65.5) PI 23.9-86.3 tau2 0.488 I2 99.4 Q 3247
noDai 63 56.3 (HK 49.1-63.3; Wald 49.3-63.1) PI 11.4-92.8 tau2 1.311 I2 99.6 Q 15740
median 24 47.5 (HK 40.2-54.8; Wald 40.6-54.4) PI 17.3-79.6 tau2 0.479 I2 99.4 Q 3647
tailpool 25 89.9 (HK 85.2-93.2; Wald 85.5-93.0) PI 50.5-98.7 tau2 1.049 I2 99.3 Q 3564
lower 24 87.5 (HK 82.0-91.6; Wald 82.3-91.4) PI 44.4-98.4 tau2 1.052 I2 99.5 Q 4583
upper 24 93.7 (HK 90.2-96.0; Wald 90.5-95.8) PI 59.2-99.3 ta

## 4. (Optional) Download outputs

In [24]:
from google.colab import files
files.download('output/meta_results.json')
files.download('output/baseline_estimates.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>